Importacion para evitar problemas de paths relativos.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

Importación e inicializacion de dependencias y servicios.

In [ ]:
import pandas as pd
from src.Orchestration.google_play_service import GooglePlayService 

google_service = GooglePlayService(lang="es", country="ar")

c:\Users\matia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13906.52it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading model: google/flan-t5-large


Loading weights: 100%|██████████| 558/558 [00:00<00:00, 4542.11it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausa

In [ ]:
import datetime

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S") #estrictamente no es un timestamp, pero es un formato de fecha y hora que se puede usar para identificar la versión de los datos descargados
app_id = "com.mercadolibre"

NameError: name 'datetime' is not defined

Primeramente el pipeline rootea los parámetros a utilizar a lo largo de su ejecucion.
Estos son:
* **app_id**: es el ID de Googlela aplicación a analizar en GooglePlay)
* **timestamp**: es el dia y hora en la cual se ejecuto el pipeline, sirve como una suerte de correlation_id para relacionar todos los archivos generados en dicha ejecución, de manera que el nombre de los archivos tendrán el formato  appid_timestamp_execution_step (whatsapp_20263181135_CLEAN).

In [ ]:

google_service.path_helper.set_filepath_components(
    app_id=app_id,
    timestamp=timestamp
)

Obtención de datos.
El servicicio se encarga de llamar a la capa que realiza la ingesta mediante la librería google-play-scraper y delegar la persistencia de los resultados en un archivo ..._*RAW*.csv 

In [ ]:
reviews_df = google_service.collect_and_store_reviews(app_id=app_id, limit=100)

print(reviews_df.head())

Ahora se procede a realizar el limpiado de los datos. El orquestador delega esta tarea al **GoogleReviewCleaner** y posteriormente realiza la persistencia de los datos ya curados. El método *clean_reviews* no genera un nuevo dataframe para evitar realizar una nueva copia, en su lugar modificia el dataframe provisto y lo retorna con dichas modificaciones, previo a persistir dichos cambios en un nuevo archivo con el prefijo ..._*CLEANED*.csv.

In [ ]:
cleaned_reviews_df = google_service.clean_reviews(reviews_df)
print(cleaned_reviews_df.head())

El siguiente paso en el pipeline es agregar el análisis de sentimientos, esto se delega a **RobertaSentimentModel**.
Nuevamente se reutiliza el dataframe para ahorrar alocación de memoria y posteriormente se guarda el archivo con su prefijo correspondiente ..._*ANALYZED*.csv

In [ ]:
sentiment_df = google_service.analyze_sentiment(cleaned_reviews_df)
print(sentiment_df.head())

Finalmente el pipeline manda a ejecutar el modelo alojado en **FlanT5LargeModel**, el cual debiera realizar el resumen de las reseñas (con su sentimiento ya asignado) y seleccionar los puntos mas positivos (negativos).
Este método sí retorna un nuevo dataframe, el cual es almacenado en un archivo ..._*SUMMARY*.csv

In [ ]:
summary_df = google_service.build_summary(sentiment_df)
print(summary_df.head())

Toda esta perorata queda resumida dentro del método *run_pipeline*. El cual ejecuta todos los pasos previos.